# Person A - day seven: the shared-environment TOFU baseline

Goal: move from artificial cache fixtures to the existing exposed TOFU questions in the validated FP32 environment. This run produces 240 ordinary-cache responses: three checkpoints x (20 forgotten + 60 fixed utility questions). It also scores the 20 reference answers per model in parallel and sequentially.

Use Colab T4. Keep the day-six environment if still available. Otherwise use a fresh runtime and the setup below. The input bundle contains the selected data; no new authors are fetched. Factual annotation follows review of the returned ZIP. The planned NPO/RMU study is unchanged.

## 1. Verify GPU selection

In [ ]:
import subprocess,sys
print(sys.version)
subprocess.run(["nvidia-smi"],check=True)


## 2. Set up packages
Run this cell. It skips installation when the declared pins already match. In a new default runtime it installs B/day-six versions. If installation occurs, **restart the session**, then continue at section 3. Removing torchvision/torchaudio avoids the import conflict observed by B. No datasets package is needed for frozen JSON inputs.

In [ ]:
import importlib.metadata,subprocess,sys
FORCE_SETUP=False
PINS={"torch":"2.9.1","transformers":"5.5.4","hqq":"0.2.8.post1","accelerate":"1.13.0","huggingface-hub":"1.33.0","numpy":"2.1.3"}
def matches(name,expected):
    try:return importlib.metadata.version(name).split("+")[0]==expected
    except importlib.metadata.PackageNotFoundError:return False
if FORCE_SETUP or not all(matches(k,v) for k,v in PINS.items()):
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision","torchaudio"],check=True)
    subprocess.run([sys.executable,"-m","pip","install","torch==2.9.1","--index-url","https://download.pytorch.org/whl/cu128"],check=True)
    subprocess.run([sys.executable,"-m","pip","install"]+[k+"=="+v for k,v in PINS.items() if k!="torch"],check=True)
    print("Setup completed. RESTART the session, then continue at section 3.")
else:print("Declared versions already match. Continue at section 3.")


## 3. Verify the environment after any required restart

In [ ]:
import torch,transformers,importlib.metadata
assert torch.cuda.is_available()
assert torch.__version__.split("+")[0]=="2.9.1" and torch.version.cuda=="12.8", "Use the CUDA 12.8 wheel in setup."
assert transformers.__version__=="5.5.4"
assert importlib.metadata.version("hqq")=="0.2.8.post1"
print(torch.cuda.get_device_name(0),torch.__version__,transformers.__version__)


## 4. Upload the day-seven preparation ZIP
Upload `day7-baseline-input-bundle.zip`. Normally leave RESUME_OUTPUT_DIR blank. To continue an interrupted run whose files still exist, enter its exact output-directory path. This requires the same bundle/environment and successful, complete prior JSONL lines. An error or broken line requires review; the runner will not silently discard it. A reset ephemeral runtime loses local files, so download the ZIP before ending a session.

In [ ]:
from google.colab import files
from pathlib import Path
from datetime import datetime,timezone
import hashlib,zipfile,io,json,sys
RESUME_OUTPUT_DIR=""
uploaded=files.upload()
NAME="day7-baseline-input-bundle.zip"
assert NAME in uploaded
payload=uploaded[NAME]
assert hashlib.sha256(payload).hexdigest()=="43778031db7e061f3ed8b5cfb32436229a7e90b83d2aba7c57e32d8b5d774253", "Notebook and bundle differ."
BASE=Path("/content/kv-day7");BASE.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SOURCE=BASE/("input_"+STAMP);SOURCE.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        assert Path(name).name==name,"Unexpected archive path."
        (SOURCE/name).write_bytes(z.read(name))
PREP=json.loads((SOURCE/"preparation-manifest.json").read_text())
for name,digest in PREP["files"].items():assert hashlib.sha256((SOURCE/name).read_bytes()).hexdigest()==digest
OUT=Path(RESUME_OUTPUT_DIR) if RESUME_OUTPUT_DIR else BASE/("day7_"+STAMP)
sys.path.insert(0,str(SOURCE))
print("Output directory:",OUT)


## 5. Run the baseline
Each checkpoint loads once in FP32. Progress prints after each completed question; responses and scores are saved immediately. Sixty forget-item parallel/sequential comparisons must pass the inherited provisional tolerances. No positive factual result is required. The 60 utility questions per model use generation only.

If this cell fails or a runtime disconnection is approaching, run section 6 to preserve partial evidence. Avoid changing checkpoints, tolerances or prompts after seeing outputs.

In [ ]:
import importlib.util,traceback
spec=importlib.util.spec_from_file_location("kv_day7_runner",SOURCE/"day7_baseline.py")
runner=importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(runner)
    report=runner.main(SOURCE,OUT,resume=bool(RESUME_OUTPUT_DIR))
    print("STATUS:",report["status"])
except Exception:
    OUT.mkdir(parents=True,exist_ok=True)
    (OUT/"notebook-error.txt").write_text(traceback.format_exc())
    raise
finally:
    if OUT.exists():(OUT/"input-bundle-sha256.txt").write_text(hashlib.sha256(payload).hexdigest()+"\n")


## 6. Download output, including partial output on failure
Send the ZIP for artifact checking and a proposed factual review. The records deliberately remain unscored. Exact old-output matches suggest labels may be reusable; changed outputs require review. All differences are configuration drift because Torch, precision and explicit cache setup differ from the earlier run.

In [ ]:
import shutil
assert OUT.exists()
archive=shutil.make_archive(str(OUT),"zip",root_dir=OUT)
print(archive)
files.download(archive)
